In [2]:
# backtesting Engine
class Event(object):
    """Base class for events."""
    pass
class MarketEvent(Event):
    """Handles the tick event."""
    def __init__(self):
        """Initialize the market event."""
        self.type = 'MARKET'
class SignalEvent(Event):
    """Handles the signal event."""
    def __init__(self, strategy_id, symbol, datetime, signal_type, strength):
        self.type = 'SIGNAL'
        self.strategy_id = strategy_id
        self.symbol = symbol
        self.datetime = datetime
        self.signal_type = signal_type
        self.strength = strength
class OrderEvent(Event):
    """Handles the order event."""
    def __init__(self, strategy_id, symbol, order_type, quantity, direction):
        self.type = 'ORDER'
        self.strategy_id = strategy_id
        self.symbol = symbol
        self.order_type = order_type
        self.quantity = quantity
        self.direction = direction
    def print_order(self):
        print("Order: Strategy ID = %s, Symbol = %s, Order Type = %s, Quantity = %s, Direction = %s" % (self.strategy_id, self.symbol, self.order_type, self.quantity, self.direction))

class FillEvent(Event):
    """Handles the fill event."""
    def __init__(self, symbol, timeindex, exchange, quantity, direction, fill_cost, commission):
        self.type = 'FILL'
        self.timeindex = timeindex
        self.symbol = symbol
        self.exchange = exchange
        self.quantity = quantity
        self.direction = direction
        self.fill_cost = fill_cost
        if commission is None:
            self.commission = commission
        else:
            self.commission = commission_ib
    def commission_ib(self):
        full_cost = 1.3
        if self.quantity <= 500:
            full_cost = max(1.3, 0.01 * self.quantity)
        else:
            full_cost = max(1.3, 0.008 * self.quantity)
        return full_cost

from abc import ABCMeta, abstractmethod 
import datetime
import os,os.path
import pandas as pd
import numpy as np

class DataHandler(object):
    """DataHandler is an abstract base class providing an interface for all subsequent (inherited) data handlers (both live and historical data handlers, as well as generic data sources and market data.)"""
    __metaclass__ = ABCMeta
    @abstractmethod
    def get_latest_bar(self,symbol):
        """returns the last bar updadted"""
        raise NotImplementedError("Should implement get_latest_bar")
    @abstractmethod
    def get_latest_bars(self,symbol,N=1):
        """returns the last N bars updadted"""
        raise NotImplementedError("Should implement get_latest_bars")
    @abstractmethod
    def get_latest_bar_datetime(self,symbol):
        """returns the last bar datetime"""
        raise NotImplementedError("Should implement get_latest_bar_datetime")

    @abstractmethod
    def get_latest_bar_value(self,symbol, val_type):
        """returns the last bar value"""
        raise NotImplementedError("Should implement get_latest_bar_value")
    
    @abstractmethod
    def get_latest_bar_values(self,symbol, val_type,N=1):
        """returns the last N bar values"""
        raise NotImplementedError("Should implement get_latest_bar_values")

    @abstractmethod
    def updadte_bars(self):
        """ updates the bars"""
        raise NotImplementedError("should implement update_bars()") 


class HistoricCSVDataHandler(DataHandler):
    """HistoricCSVDataHandler is a class for handling historical data from a CSV file."""
    def __init__(self,events,csv_dir,symbol_list):
        """Initialize the data handler.
        Initialises the historic data handler by requesting
        the location of the CSV files and a list of symbols.
        It will be assumed that all files are of the form
        'symbol.csv', where symbol is a string in the list.
        Parameters:
        events - The Event Queue.
        csv_dir - Absolute directory path to the CSV files.
        symbol_list - A list of symbol strings."""

        self.events = events
        self.csv_dir = csv_dir
        self.symbol_list = symbol_list
        self.symbol_data = {}
        self.latest_symbol_data={}
        self.continue_backtest =True
        self._open_covert_csv_files()
    def _open_covert_csv_files(self):
        comb_index =None
        for s in self.symbol_list:
            self.symbol_data[s] = pd.read_csv(os.path.join(self.csv_dir, '%s.csv' % s),header=0,index_col=0,parse_dates=True,names=["datetime","open","high","low","close","volume"]).sort()
            if comb_index is None:
                comb_index=self.symbol_data[s].index
            else:
                comb_index.union(self.symbol_data[s].index)
            self.latest_symbol_data[s]=[]
        for s in self.symbol_list:
            self.symbol_data[s]=self.symbol_data[s].\
                reindex(index=comb_index,method='pad').iterrows()
    def _get_new_bar(self,symbol):
        for b in self.symbol_data[symbol]:
            yield b
    def get_latest_bar(self,symbol):
        try:
            bars_list=self.latest_symbol_data[symbol]
        except KeyError:
            print("No data found for symbol: %s" % symbol)
            raise

        else:
            return bars_list[-1]

    def get_latest_bars(self,symbol,N=1):
        try:
            bars_list=self.latest_symbol_data[symbol]
        except KeyError:
            print("No data found for symbol: %s" % symbol)
            raise
        else:
            return bars_list[-N:]

    def get_latest_bar_datetime(self,symbol):
        try:
            bars_list=self.latest_symbol_data[symbol]
        except KeyError:
            print("No data found for symbol: %s" % symbol)
            raise
        else:
            return bars_list[-1][0] 

    def get_latest_bar_value(self,symbol,val_type):
        try:
            bars_list=self.latest_symbol_data[symbol]
        except KeyError:
            print("No data found for symbol: %s" % symbol)
            raise
        else:
            return getattr(bars_list[-1][1],val_type)
    
    def get_latest_bar_values(self,symbol,val_type,N=1):
        try:
            bars_list=self.get_latest_bars(symbol,N)
        except KeyError:
            print("No data found for symbol: %s" % symbol)
            raise
        else:
            return np.array([getattr(bars_list[-1][1],val_type) for b in bars_list])

    def updadate_bars(self):
        for s in self.symbol_list:
            try:
                bar=next(self._get_new_bar(s))
            except StopIteration:
                self.continue_backtest=False
            else:
                if bar is not None:
                    self.events.put(bar)
                    self.latest_symbol_data[s].append(bar)    
        self.events.put(MarketEvent())
try:
    import Queue as queue
except ImportError:
    import queue

class Strategy(object):
    """Strategy is an abstract base class providing an interface for all subsequent (inherited) strategy handling objects."""
    __metaclass__ = ABCMeta
    @abstractmethod
    def calculate_signals(self):
        """Provides the mechanisms to calculate the list of signals."""
        raise NotImplementedError("Should implement calculate_signals()")

def create_sharpe_ratio(returns,period=252):
    return np.sqrt(period)*np.mean(returns)/np.std(returns) 

def create_drawdowns(pnl):
    hwm = np.zeros(pnl.shape)
    for t in range(hwm.shape[0]):
        hwm[t] = max(hwm[t-1],pnl[t])
        drawdown[t]=(hwm[t]-pnl[t])
        duration[t]=(0 if drawdown[t]==0 else duration[t-1]+1)
    return drawdown,dawdown.max(),duration.max()

class Portfolio(object):
    """Portfolio is an abstract base class providing an interface for all subsequent (inherited) portfolio objects."""
    def __init__(self,events,symbol_list,initial_capital=100000.0):
        self.bars= bars 
        self.events = events
        self.symbol_list = symbol_list
        self.initial_capital = initial_capital
        self.all_positions=self.construct_all_positions()
        self.current_positions= dict((k,v) for k,v in [(s,0) for s in self.symbol_list])

        self.all_positions = self.construct_all_positions()
        self.current_holdings = self.construct_current_holdings()
    
    def construct_all_positions(self):
        d= dict((k,v) for k,v in [(s,0) for s in self.symbol_list])
        d["datetime"]=self.start_date
        return d
    
    def construct_all_holdings(self):
        d= dict((k,v) for k,v in [(s,0) for s in self.symbol_list])
        d["datetime"]=self.start_date
        d["cash"] = self.initial_capital
        d["commission"] = 0.0
        d["total"]=self.initial_capital
        return d
    
    def contruct_current_holdings(self):
        d= dict((k,v) for k,v in [(s,0) for s in self.symbol_list])
        d["cash"] = self.initial_capital
        d["commission"] = 0.0
        d["total"]=self.initial_capital
        return d
    
    def update_timeindex(self):
        """Adds a new record to the positions matrix for the current
            market data bar. This reflects the PREVIOUS bar, i.e. all
            current market data at this stage is known (OHLCV).
            Makes use of a MarketEvent from the events queue."""
        latest_datetime = self.bars.get_latest_bar_datetime(self.symbol_list[0])
        dp=dict((k,v) for k,v in [(s,0) for s in self.symbol_list])
        dp["datetime"] = latest_datetime
        for s in self.symbol_list:
            dp[s]=self.current_positions[s]
        self.all_positions.append(dp)
        dh=dict((k,v) for k,v in [(s,0) for s in self.symbol_list])
        dh["datetime"] = latest_datetime
        dh["cash"] = self.current_holdings["cash"]
        dh["commission"] = self.current_holdings["commission"]
        dh["total"] = self.current_holdings["cash"]
        
        for s in self.symbol_list:
            market_value = self.current_holdings[s]*self.bars.get_latest_bar_value(s,"close")   
            dh[s] = market_value
            dh["total"] += market_value
            
        self.all_holdings.append(dh)

    def update_positions_from_fill(self,fill):
        """Updates the portfolio current positions based on a fill."""
        fill_dir =0
        if fill.direction=="BUY":
            fill_dir=1
        if fill.direction=="SELL":
            fill_dir=-1
        self.current_positions[fill.symbol]+=fill_dir*fill.quantity


    def update_holdings_from_fill(self,fill):
        """Updates the portfolio current holdings based on a fill."""
        fill_dir =0
        if fill.direction=="BUY":
            fill_dir=1
        if fill.direction=="SELL":
            fill_dir=-1
        # Update holdings list with new quantities
        fill_cost=self.bars.get_latest_bar_value(fill.symbol,"adj_close")
        
        cost =fill_dir*fill_cost*fill.quantity
        self.current_holdings[fill.symbol]+=cost
        self.current_holdings["commission"]+=fill.commission
        self.current_holdings["cash"]-=(cost+fill.commission)
        self.current_holdings["total"]-=(cost+fill.commission)

    def update_fill(self,event):
        """Updates the portfolio current holdings based on a fill."""
        if event.type=='FILL':
            self.update_positions_from_fill(event)
            self.update_holdings_from_fill(event)

    def generate_naive_order(self,signal):
        """Generates the orders assuming 100% fill."""
        order = None
        symbol=signal.symbol
        direction=signal.signal_type
        quantity=signal.quantity
        strength=signal.strength
        mkt_quantity=100
        cur_quantity=self.current_holdings[symbol]
        order_type='MKT'
        if direction=='LONG' and cur_quantity==0:
            order=OrderEvent(symbol,order_type,mkt_quantity,'BUY')
        if direction=='SHORT' and cur_quantity==0:
            order=OrderEvent(symbol,order_type,mkt_quantity,'BUY')
        if direction=='EXIT' and cur_quantity>0:
            order=OrderEvent(symbol,order_type,abs(cur_quantity),'SELL')
        if direction=='EXIT' and cur_quantity<0:
            order=OrderEvent(symbol,order_type,abs(cur_quantity),'BUY')
        return order
    def update_signal(self,event):
        if event.type=='SIGNAL':
            order_event=self.generate_naive_order(event)
            self.events.put(order_event)

    def create_equity_curve_dataframe(self):
        curve = pd.DataFrame(self.all_holdings)
        curve.set_index('datetime',inplace=True)
        curve['returns'] = curve['total'].pct_change()
        curve['equity_curve'] = (1+curve['returns']).cumprod()
        curve['equity_curve'].plot(figsize=(10,5))
        plt.title('Equity Curve')
        plt.show()  
        return curve
        
    def output_summary_stats(self):
        total_return = self.equity_curve[-1]
        returns =self.equity_curve['returns']
        pnl = self.equity_curve['equity_curve']
        sharpe_ratio = create_sharpe_ratio(returns,periods=252*60*6.5)
        drawdown ,max_dd,dd_duration = create_drawdowns(pnl)
        self.equity_curve['drawdown']=drawdown
        stats = [("Total Return", "%0.2f%%" % \
                    ((total_return - 1.0) * 100.0)),
                    ("Sharpe Ratio", "%0.2f" % sharpe_ratio),
                    ("Max Drawdown", "%0.2f%%" % (max_dd * 100.0)),
                    ("Drawdown Duration", "%d" % dd_duration)]
        self.equity_curve.plot(figsize=(10,5))
        plt.title('Equity Curve')
        plt.show()  
        return stats

class ExecutionHandler(ABCMeta):
    @abstractmethod
    def execute_order(self,event):
        raise NotImplementedError("Should implement execute_order()")

class SimulatedExecutionHandler(ExecutionHandler):
    def __init__(self,events):
        self.events=events
    
    def execute_order(self,event):
        if event.type=='ORDER':
            fill_event=FillEvent(event.symbol,datetime.datetime.utcnow(),event.order_type,event.quantity,event.direction)
            self.events.put(fill_event)

In [1]:
from ib_insync import IB, Stock, Order
class IBExecutionHandler(ExecutionHandler):
    def __init__(self,events):
        self.events=events
        self.order_routing=order_routing
        self.currency=currency
        self.fill_dict={}
        self.tws_conn=self.create_tws_connection()
        self.order_id=self.create_initial_order_id()
        self.register_handlers()
    
    def _error_handler(self,msg):
        if msg.error.code()==1101:
            print("WARNING: Could not connect to TWS")
        else:
            print("Error: %s" % msg.error.message())
    
    def _reply_handler(self,msg):
        if msg.typeName=="openOrder" and \
            msg.orderId==self.order_id and \
            not self.fill_dict.has_key(msg.orderId):
            self.create_fill_dict_entry(msg)
        if msg.typeName == "orderStatus" and \
            msg.status == "Filled" and \
            self.fill_dict[msg.orderId]["filled"] == False:
            self.create_fill(msg)
            print("Server Response: %s, %s\n" % (msg.typeName, msg))
    
    def create_tws_connection(self):
        """Connect to the Trader Workstation (TWS) running on the
        usual port of 7496, with a clientId of 10.
        The clientId is chosen by us and we will need
        separate IDs for both the execution connection and
        market data connection, if the latter is used elsewhere."""
        tws_conn =ibConnection()
        tws_conn.connect()
        return tws_conn
    
    def create_initial_order_id(self):
        """Create an initial order ID."""
        return 1000
    
    def register_handlers(self):
        self.tws_conn.register(self._error_handler, 'Error')
        self.tws_conn.register(self._reply_handler, 'reply')

    def create_contract(self,symbol,sec_type,exch,prim_exch, currency):
        """Create a contract object for the given symbol.
        symbol - The ticker symbol for the contract
        sec_type - The security type for the contract (’STK’ is ’stock’)
        exch - The exchange to carry out the contract on
        prim_exch - The primary exchange to carry out the contract on
        currency - The currency in which to purchase the contract"""
        contract = Contract()
        contract.m_symbol = symbol
        contract.m_secType = sec_type
        contract.m_exchange = exch
        contract.m_primaryExchange = prim_exch
        contract.m_currency = currency
        return contract

    def create_order(self,order_type,quantity,direction):
        """Create an order object for the given order type, quantity and direction."""
        order = Order()
        order.m_orderType = order_type
        order.m_totalQuantity = quantity
        order.m_action = direction
        return order

    def create_fill_dict_entry(self,msg):
        """Create a fill dictionary entry for the given order ID."""
        self.fill_dict[msg.orderId] = {
            "symbol": msg.contract.m_symbol,
            "exchange": msg.contract.m_exchange,
            "direction":msg.order.m_action,
            "filled":False
        }
    
    def create_fill(self,msg):
        """Create a fill event for the given order ID."""
        fd=self.fill_dict[msg.orderId]
        symbol=fd["symbol"]
        exchange=fd["exchange"]
        direction=fd["direction"]
        fill_cost=msg.avgFillPrice
        fill_event=FillEvent(symbol,datetime.datetime.utcnow(),direction,exchange,fill_cost)
        self.fill_dict[msg.orderId]["filled"] = True
        self.events.put(fill_event)

    def execute_order(self,contract,order):
        """Execute the given order on the contract."""
        if event.type=="OrderEvent":
            asset=event.symbol
            asset_type="STK"
            order_type=event.order_type
            quantity=event.quantity
            direction=event.direction

            ib_contract=self.create_contract(asset,asset_type,self.order_routing,self.order_routing,self.currency)
            ib_order=self.create_order(order_type,quantity,direction)
            self.tws_conn.placeOrder(ib_contract,ib_order)
            time.sleep(1)
            self.order_id+=1


NameError: name 'ExecutionHandler' is not defined

In [ ]:
class Backtest(object):
    def __init__(self, symbol, initial_capital, heartbeat, start_date, data_handler, execution_handler, portfolio, strategy):
        self.symbol = symbol
        self.initial_capital = initial_capital
        self.heartbeat = heartbeat
        self.start_date = start_date
        self.data_handler = data_handler
        self.execution_handler = execution_handler
        self.portfolio = portfolio
        self.strategy = strategy
        self.events = queue.Queue()
        self.backtest = Backtest(self.events, self.data_handler, self.execution_handler, self.portfolio, self.strategy)
        self.backtest.simulate_trading()